# SAC Training: NBV Head = RL Agent

Этот ноутбук обучает **NBV Head модели ODIN** как RL-агента (политику) алгоритмом **SAC**.

**Ключевая идея:** NBV Head — это и есть политика. Она предсказывает следующую лучшую позицию камеры.
Coverage Head даёт сигнал награды (снижение неопределённости p_hidden).
Отдельного SAC-агента (SB3) **нет** — вся политика живёт внутри ODIN.

**Датасет сцен НЕ нужен** — среда PyBullet генерирует сцены динамически.

**Kaggle Inputs (подключить перед запуском):**
- Веса ODIN: загрузить `model_final.pth` как Kaggle Dataset (например `nbv-odin-weights`)
- *(Опционально)* Предыдущие checkpoints для resume

**Порядок запуска:** ячейки 1 → 2 → 3 → 4

## 1. Установка зависимостей (venv + ODIN стек)

In [ ]:
import os
import subprocess
import sys
import urllib.request

CONFIG = {
    "ODIN_DIR": "my_odin",
    "ODIN_REPO_URL": "https://github.com/SergKurchev/my_odin.git",
    "ODIN_BRANCH": "feature/nbv_dataset_process",
    "RL_REPO_URL": "https://github.com/SergKurchev/article-nbv.git",
    "RL_DIR": "nbv_rl",
    "ODIN_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/scannet_resnet_47.8_73.3_32k_1.5k.pth",
    "ODIN_WEIGHTS_PATH": "my_odin/models/odin_scannet_context.pth",
    "M2F_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/m2f_coco.pkl",
    "M2F_WEIGHTS_PATH": "my_odin/models/model_final_5c90d4.pkl",
}

# Создаём venv
if not os.path.exists("venv"):
    subprocess.run(["apt-get", "update", "-y"], check=False)
    subprocess.run(["apt-get", "install", "-y", "python3.10", "python3.10-venv",
                    "python3.10-dev", "python3.10-distutils",
                    "libgl1", "libglib2.0-0"], check=False)
    subprocess.run(["python3.10", "-m", "venv", "venv", "--without-pip"], check=True)
    urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", "get-pip.py")
    subprocess.run(["venv/bin/python", "get-pip.py"], check=True)
    os.remove("get-pip.py")
    print("venv created")

VENV_PYTHON = os.path.abspath("venv/bin/python")
VENV_PIP = os.path.abspath("venv/bin/pip")

def make_venv_env(extra=None):
    env = os.environ.copy()
    venv_dir = os.path.abspath("venv")
    env["VIRTUAL_ENV"] = venv_dir
    env["PATH"] = os.path.join(venv_dir, "bin") + ":" + env.get("PATH", "")
    env.pop("PYTHONPATH", None)
    if extra:
        env.update(extra)
    if "RL_DIR" in CONFIG:
        env["PYTHONPATH"] = os.path.abspath(CONFIG["RL_DIR"]) + ":" + os.path.abspath(CONFIG["ODIN_DIR"])
    return env

def run_cmd(cmd, cwd=None, env=None, check=True):
    if cmd[0] == "pip": cmd[0] = VENV_PIP
    elif cmd[0] == "python": cmd[0] = VENV_PYTHON
    print(f">>> {' '.join(str(c) for c in cmd)}")
    subprocess.run(cmd, cwd=cwd, env=env, check=check)

def clean_build(directory):
    import shutil, glob
    for d in ["build", "dist"]:
        path = os.path.join(directory, d)
        if os.path.exists(path): shutil.rmtree(path)
    for egg in glob.glob(os.path.join(directory, "*.egg-info")):
        shutil.rmtree(egg)

def install_all():
    venv_env = make_venv_env()
    cuda_env = make_venv_env({"FORCE_CUDA": "1", "TORCH_CUDA_ARCH_LIST": "6.0;7.0;7.5;8.0;8.6"})

    # 0. Клонируем ODIN
    if not os.path.exists(CONFIG["ODIN_DIR"]):
        subprocess.run(["git", "clone", "-q", "-b", CONFIG["ODIN_BRANCH"],
                        CONFIG["ODIN_REPO_URL"], CONFIG["ODIN_DIR"]], check=True)

    # 0b. Клонируем RL репозиторий (article-nbv)
    if not os.path.exists(CONFIG["RL_DIR"]):
        subprocess.run(["git", "clone", "-q", CONFIG["RL_REPO_URL"], CONFIG["RL_DIR"]], check=True)

    # 1. PyTorch 2.2.0 + CUDA 12.1
    print("\n1. Installing PyTorch...")
    run_cmd(["pip", "install", "-q", "torch==2.2.0", "torchvision==0.17.0",
             "--index-url", "https://download.pytorch.org/whl/cu121"], env=venv_env)
    run_cmd(["pip", "install", "-q", "torch-scatter",
             "-f", "https://data.pyg.org/whl/torch-2.2.0+cu121.html"], env=venv_env)

    # 2. NumPy + Pillow
    print("\n2. Installing NumPy + Pillow...")
    run_cmd(["pip", "install", "-q", "numpy<2", "--force-reinstall"], env=venv_env)
    run_cmd(["pip", "install", "-q", "Pillow>=10.2.0"], env=venv_env)

    # 3. Фильтрация requirements.txt
    print("\n3. Cleaning ODIN requirements...")
    req_path = os.path.join(CONFIG["ODIN_DIR"], "requirements.txt")
    with open(req_path, 'r') as f: lines = f.readlines()
    with open(req_path, 'w') as f:
        for line in lines:
            lc = line.strip().lower()
            if any(x in lc for x in ["waspinator", "detectron2", "pytorch3d"]): continue
            if "pyyaml==5.3.1" in lc: f.write("pyyaml>=5.4.1\n")
            else: f.write(line)

    # 4. Build tools
    print("\n4. Build tools...")
    run_cmd(["pip", "install", "-q", "cython", "setuptools", "wheel", "pycocotools"], env=venv_env)

    # 5. ODIN requirements
    print("\n5. ODIN requirements...")
    run_cmd(["pip", "install", "-q", "-r", req_path], env=venv_env)
    run_cmd(["pip", "install", "-q", "ninja", "fvcore", "iopath"], env=venv_env)

    # 6. Detectron2
    print("\n6. Detectron2...")
    run_cmd(["pip", "install", "-q", "--no-build-isolation",
             "git+https://github.com/facebookresearch/detectron2.git"], env=venv_env)

    # 7. PyTorch3D
    print("\n7. PyTorch3D...")
    run_cmd(["pip", "install", "-q", "--no-build-isolation",
             "git+https://github.com/facebookresearch/pytorch3d.git"], env=cuda_env)

    # 8. NumPy + OpenCV pin
    print("\n8. Pinning NumPy + OpenCV...")
    run_cmd(["pip", "uninstall", "-y", "-q", "numpy"], env=venv_env)
    run_cmd(["pip", "install", "-q", "numpy==1.26.4"], env=venv_env)
    run_cmd(["pip", "install", "-q", "opencv-python-headless==4.8.0.76"], env=venv_env)

    # 9. pointops2 CUDA kernel
    print("\n9. pointops2...")
    pointops_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "libs", "pointops2"))
    clean_build(pointops_dir)
    run_cmd(["python", "setup.py", "install"], cwd=pointops_dir, env=cuda_env)

    # 10. Deformable attention
    print("\n10. Deformable attention...")
    deform_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "odin", "modeling", "pixel_decoder", "ops"))
    clean_build(deform_dir)
    run_cmd(["python", "setup.py", "build", "install"], cwd=deform_dir, env=cuda_env)

    # 11. RL зависимости
    print("\n11. RL dependencies (pybullet, gymnasium)...")
    run_cmd(["pip", "install", "-q",
             "pybullet", "gymnasium",
             "imageio", "pandas", "matplotlib"], env=venv_env)

    print("\n=== Installation complete ===")

install_all()

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,061 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
libgl1 is already the newest version (1.4.0-1).
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
python3-distutils is already the newest version (3.10.8-1~22.04).
python3-distutils set to manually installed.
The following additional packages will be installed:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3-pip-whl python3-setuptools-whl python3.10-minimal
Suggested packages:
  python3.10-doc binfmt-support
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.10-dev python3.10-venv
The following packages will be upgraded:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3.10 python3.10-minimal
6 upgraded, 4 newly installed, 0 to remove and 218 not upgraded.
Need to get 15.2 MB of archives.
After this operation, 3,272 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu ja

## 2. Поиск весов ODIN (датасет сцен НЕ нужен)

In [9]:
ODIN_CFG = "my_odin/configs/scannet_context/3d.yaml"

# Ищем предобученные веса NBVActiveODIN
# Загрузите model_final.pth как Kaggle Dataset (напр. 'nbv-odin-weights')
POSSIBLE_WEIGHTS = [
    "/kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth",
    "/kaggle/input/nbv-odin-weights/model_final.pth",  # Kaggle Dataset
    "./output_nbv_stage2/model_final.pth",              # Из предыдущего запуска
    "./output_nbv_stage2/last_checkpoint.pth",
    "./output_odin_sac/best.pth",                       # Предыдущий SAC запуск
]

ODIN_WEIGHTS = None
for p in POSSIBLE_WEIGHTS:
    if os.path.exists(p):
        ODIN_WEIGHTS = p
        print(f"✓ Found ODIN weights: {p}")
        break

if ODIN_WEIGHTS is None:
    print("⚠ NBVActiveODIN weights not found! Downloading base ODIN weights...")
    ODIN_WEIGHTS = CONFIG["ODIN_WEIGHTS_PATH"]
    os.makedirs(os.path.dirname(ODIN_WEIGHTS), exist_ok=True)
    if not os.path.exists(ODIN_WEIGHTS):
        os.system(f"wget --tries=3 -q '{CONFIG['ODIN_WEIGHTS_URL']}' -O '{ODIN_WEIGHTS}'")

print(f"\nODIN_WEIGHTS = {ODIN_WEIGHTS}")
print(f"ODIN_CFG     = {ODIN_CFG}")
print(f"\nДатасет сцен НЕ нужен — PyBullet генерирует сцены динамически.")

✓ Found ODIN weights: /kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth

ODIN_WEIGHTS = /kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth
ODIN_CFG     = my_odin/configs/scannet_context/3d.yaml

Датасет сцен НЕ нужен — PyBullet генерирует сцены динамически.


## 3. Синхронизация checkpoint (для продолжения обучения)

In [10]:
import shutil

OUTPUT_DIR = "./output_odin_sac"

def sync_previous_output(target_output=OUTPUT_DIR):
    """Копирует старые checkpoints из /kaggle/input."""
    os.makedirs(target_output, exist_ok=True)
    for root, dirs, files in os.walk("/kaggle/input"):
        if "output_odin_sac" in root or "output_rl" in root:
            pth_files = [f for f in files if f.endswith(".pth")]
            if pth_files:
                print(f"Found previous checkpoint in: {root}")
                for f in pth_files:
                    src = os.path.join(root, f)
                    dst = os.path.join(target_output, f)
                    if not os.path.exists(dst):
                        shutil.copy2(src, dst)
                        print(f"  Copied: {f}")
                return
    print("No previous checkpoints found. Training from scratch.")

sync_previous_output()

# Проверяем наличие best/last для информации
for f in ["best.pth", "last.pth"]:
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(path):
        print(f"✓ Found: {path}")

No previous checkpoints found. Training from scratch.
✓ Found: ./output_odin_sac/best.pth


## 4. Запуск обучения: SAC + NBV Head как RL Agent

NBV Head = Actor (политика). Coverage Head = Reward signal.

**`--freeze_backbone`** замораживает backbone ODIN + Coverage Head.
Обучается **только NBV Head** (+ Twin Q-critics для SAC).

In [11]:
# 1

In [16]:
!cd /kaggle/working/my_odin && git pull


Already up to date.


In [17]:
!cd nbv_rl && git pull origin master


remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 315 bytes | 315.00 KiB/s, done.
From https://github.com/SergKurchev/article-nbv
 * branch              master     -> FETCH_HEAD
   803e0703..01f7b18d  master     -> origin/master
Updating 803e0703..01f7b18d
Fast-forward
 config.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [18]:
# import os
# import subprocess

# def check_repo_commit(repo_dir, expected_hash, expected_msg):
#     print(f"=== Проверка репозитория: {repo_dir} ===")
#     if not os.path.exists(repo_dir):
#         print(f"❌ Папка {repo_dir} НЕ СУЩЕСТВУЕТ! (Нужно запустить git clone/install_all)\n")
#         return

#     try:
#         # Получаем хеш и сообщение последнего коммита
#         commit_hash = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir).decode("utf-8").strip()
#         commit_msg = subprocess.check_output(["git", "log", "-1", "--pretty=%B"], cwd=repo_dir).decode("utf-8").strip()
        
#         print(f"📌 Текущий коммит: {commit_hash}")
#         print(f"📝 Сообщение:      {commit_msg.splitlines()[0]}")
        
#         if commit_hash.startswith(expected_hash):
#             print("✅ Всё супер! Загружен правильный актуальный коммит.\n")
#         else:
#             print(f"⚠️ ВНИМАНИЕ: Загружен старый коммит! Ожидался: {expected_hash} ({expected_msg})")
#             print(f"👉 Выполни очистку: shutil.rmtree('{repo_dir}') и перезапусти клонирование!\n")
#     except Exception as e:
#         print(f"❌ Ошибка при проверке git в {repo_dir}: {e}\n")

# # Проверяем RL репозиторий (article-nbv)
# check_repo_commit("nbv_rl", "4067eda", "Add rl_mode flag in odin_adapter.py")

# # Проверяем ODIN репозиторий (my_odin)
# check_repo_commit("my_odin", "e0087c2", "Add rl_mode check in NBVActiveODIN.forward")


In [19]:
RL_DIR = os.path.abspath(CONFIG["RL_DIR"])
ODIN_DIR = os.path.abspath(CONFIG["ODIN_DIR"])

RL_DIR = os.path.abspath(CONFIG["RL_DIR"])
ODIN_DIR = os.path.abspath(CONFIG["ODIN_DIR"])

# ====================================================================
# Конфигурация обучения — редактируйте параметры здесь
# ====================================================================
TOTAL_STEPS      = 100000   # Общее число шагов SAC
MAX_STEPS        = 5        # Максимальное число шагов в эпизоде
LR_ACTOR         = "1e-4"   # Learning rate NBV Head (actor)
LR_CRITIC        = "3e-4"   # Learning rate Q-networks (critic)
BUFFER_SIZE      = "50000"  # Размер Replay Buffer
BATCH_SIZE       = "256"    # Batch для SAC updates
LEARNING_STARTS  = 1000     # Шагов до начала обучения (int для расчетов)
SCENE_STAGE      = "2"      # 1=single obj, 2=multi obj, 3=multi+obstacles
FREEZE_BACKBONE  = True     # True: только NBV Head. False: end-to-end
TRAIN_LAST_TRANSFORMER_BLOCK = True  # True: train last block of transformer in backbone
# ====================================================================

# Переводим контрольные точки из шагов (steps) в номера эпизодов (episodes)
ep_start = LEARNING_STARTS // MAX_STEPS
ep_half  = (TOTAL_STEPS // 2) // MAX_STEPS
ep_total = TOTAL_STEPS // MAX_STEPS

record_eps = [
    10, 11, 12, 13, 14, 15, 50, 100,
    ep_start, ep_start+1, ep_start+2, ep_start+3, ep_start+4,        # 200-й эпизод (1k шагов)
    ep_start * 2, ep_start * 2 + 1, ep_start * 2 + 2, ep_start * 2 + 3, ep_start * 2 + 4,    # 400-й эпизод (2k шагов)
    ep_start * 3,    # 600-й эпизод (3k шагов)
    ep_start * 4,    # 800-й эпизод (4k шагов)
    ep_start * 5, ep_start * 5 + 1, ep_start * 5 + 2, ep_start * 5 + 3, ep_start * 5 + 4,    # 1000-й эпизод (5k шагов)
    ep_half,         # 10 000-й эпизод (50k шагов)
    ep_total         # 20 000-й эпизод (100k шагов)
]
# Преобразуем все числа в строки для subprocess
record_eps_str = [str(ep) for ep in record_eps]

train_cmd = [
    VENV_PYTHON,
    f"{RL_DIR}/train_odin_sac_rl.py",

    # --- Веса и конфиг ODIN ---
    "--odin_weights", ODIN_WEIGHTS,
    "--odin_cfg", ODIN_CFG,

    # --- Параметры SAC ---
    "--total_steps", str(TOTAL_STEPS),
    "--max_steps", str(MAX_STEPS),  
    "--lr_actor", LR_ACTOR,
    "--lr_critic", LR_CRITIC,
    "--buffer_size", BUFFER_SIZE,
    "--batch_size", BATCH_SIZE,
    "--learning_starts", str(LEARNING_STARTS),
    "--record_episodes"
] + record_eps_str + [

    # --- Конфигурация сцены ---
    "--scene_stage", SCENE_STAGE,
    "--num_classes", "24",

    # --- Output ---
    "--output_dir", OUTPUT_DIR,
    "--save_freq", "5000",
    "--log_freq", "10",
]

# Флаг заморозки backbone
if FREEZE_BACKBONE:
    train_cmd.append("--freeze_backbone")
if TRAIN_LAST_TRANSFORMER_BLOCK:
    train_cmd.append("--train_last_transformer_block")

# PYTHONPATH: RL repo + ODIN repo
venv_env = make_venv_env({
    "PYTHONPATH": f"{RL_DIR}:{ODIN_DIR}",
    "PYBULLET_HEADLESS": "1",
    "DISPLAY": "",
})

print("=" * 60)
print("  SAC Training: NBV Head = RL Agent (Actor)")
print(f"  ODIN weights: {ODIN_WEIGHTS}")
print(f"  Total steps:  {TOTAL_STEPS}")
print(f"  Max steps/ep: {MAX_STEPS}")
print(f"  Record eps:   {record_eps}")
print(f"  Freeze:       {FREEZE_BACKBONE}")
print(f"  Scene stage:  {SCENE_STAGE}")
print("=" * 60)
print()

subprocess.run(train_cmd, env=venv_env, check=True)

  SAC Training: NBV Head = RL Agent (Actor)
  ODIN weights: /kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth
  Total steps:  100000
  Max steps/ep: 5
  Record eps:   [10, 11, 12, 13, 14, 15, 50, 100, 200, 201, 202, 203, 204, 400, 401, 402, 403, 404, 600, 800, 1000, 1001, 1002, 1003, 1004, 10000, 20000]
  Freeze:       True
  Scene stage:  2



[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
/kaggle/working/venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
pybullet build time: Jan 29 2025 23:16:28


[INFO] Loading ODIN from /kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth ...
8
8
8
output_norm GroupNorm(32, 256, eps=1e-05, affine=True)
[INFO] --freeze_backbone: trainable 1,671,304 params (NBV Head + Coverage Head + Last Transformer Block 8)
[INFO] Loaded history: 198 episodes. Next episode: 198

  SAC Training: NBV Head as RL Policy (Actor)
  Steps: 100000 | γ=0.99 | freeze=True

>>> ODIN: Standard Unity Y-flip applied to backprojection.
>>> ODIN Debug: Poses mean: 0.1935
>>> ODIN Debug: Poses shape: torch.Size([1, 1, 4, 4])
>>> ODIN Debug: World XYZ range: min=-0.64, max=1.35
Number of frames: 1
Number of frames: 1
Number of frames: 2
Number of frames: 2
Number of frames: 3
Number of frames: 3
Number of frames: 4
Number of frames: 4
Number of frames: 5
Number of frames: 5
Number of frames: 1
Number of frames: 1
Number of frames: 2
Number of frames: 2
Number of frames: 3
Number of frames: 3
Number of frames: 4
N

KeyboardInterrupt: 

## Результаты

Файлы сохраняются в `./output_odin_sac/`:

| Файл | Описание |
|------|----------|
| `best.pth` | Лучшая политика (NBV Head + Q-networks) |
| `last.pth` | Последний checkpoint |
| `ckpt_step*.pth` | Промежуточные checkpoints (каждые 5000 шагов) |
| `sac_metrics.csv` | Метрики: reward, p_hidden, critic/actor loss, alpha |
| `logs/training_metrics.csv` | Пошаговые метрики среды |

### Что содержит checkpoint (.pth)
```python
state = torch.load('best.pth')
state['nbv_head']   # Веса обученного NBV Head (= политика)
state['critic']     # Twin Q-networks
state['log_std']    # Learnable exploration noise
state['log_alpha']  # Entropy coefficient
```

### Архитектура
```
ODIN Backbone (frozen) → scene features
                              ↓
       NBV Head (Actor) → next_camera_pose   ← обучается SAC
       Coverage Head    → p_hidden           ← сигнал награды
       Twin Q-Critics   → Q(s, a)            ← обучается SAC
```